In [1]:
from dotenv import load_dotenv
import anthropic
import json
from datetime import datetime
from anthropic.types import ToolParam

load_dotenv()
client = anthropic.Anthropic()
model = "claude-opus-5-5"
max_tokens = 10000

# Visualize Utilities

In [2]:
from pydantic import BaseModel

def _short(s, n=300):
    s = str(s)
    return s if len(s) <= n else s[:n] + f"… (+{len(s) - n} chars)"

def _block_line(b):
    """Format one content block (SDK object or dict) as a single readable line."""
    get = b.get if isinstance(b, dict) else lambda k, d=None: getattr(b, k, d)
    t = get("type")
    if t == "text":
        return f"[text] {_short(get('text'))}"
    if t == "thinking":
        body = get("thinking") or "(hidden - display omitted)"
        return f"[thinking] {_short(body, 200)}  sig={len(get('signature') or '')} chars"
    if t == "tool_use":
        return f"[tool_use] {get('name')}({json.dumps(get('input'))})  id={get('id')}"
    if t == "tool_result":
        err = "  ERROR" if get("is_error") else ""
        return f"[tool_result] for {get('tool_use_id')}{err}: {_short(get('content'))}"
    return f"[{t}] {_short(b)}"

def _is_nested(v):
    return isinstance(v, BaseModel) or (isinstance(v, (list, dict)) and len(v) > 0)

def _walk(value, path, rows):
    """Collect (python access path, value) rows for every field, recursing into objects, lists and dicts."""
    if isinstance(value, BaseModel):
        rows.append((path, f"# {type(value).__name__}"))
        # Simple fields first ('type' leading), nested objects/lists after.
        names = sorted(type(value).model_fields,
                       key=lambda n: (_is_nested(getattr(value, n)), n != "type"))
        for name in names:
            _walk(getattr(value, name), f"{path}.{name}", rows)
    elif isinstance(value, list) and value:
        for i, item in enumerate(value):
            _walk(item, f"{path}[{i}]", rows)
    elif isinstance(value, dict) and value:
        for k, v in value.items():
            _walk(v, f'{path}["{k}"]', rows)
    else:
        rows.append((path, "= " + _short(repr(value), 80)))

def show_response(response, full=True, name="response"):
    """Print a Message. full=True lists every field as the Python expression that reads it."""
    if not full:
        u = response.usage
        print(f"── {response.id}  model={response.model}")
        print(f"   stop_reason = {response.stop_reason}")
        print(f"   tokens: in={u.input_tokens}  out={u.output_tokens}"
              f"  cache_read={u.cache_read_input_tokens or 0}")
        for i, b in enumerate(response.content):
            print(f"   {i}. {_block_line(b)}")
        return

    rows = []
    _walk(response, name, rows)
    width = max(len(path) for path, _ in rows) + 2
    for path, text in rows:
        # Blank line before each content block and before usage (objects one level below the response).
        if text.startswith("#") and path.count(".") == 1:
            print()
        print(f"{path:<{width}}{text}")

def show_history(messages):
    for i, m in enumerate(messages):
        content = m["content"]
        if isinstance(content, str):
            content = [{"type": "text", "text": content}]
        print(f"{i:>2} {m['role']:<9}  " + f"\n{'':>14}".join(_block_line(b) for b in content))

## Functions

In [3]:
def create_user_text_message(text):
    content = {"type": "text", "text": text}
    message = {"role": "user", "content": [content]}
    return message

def create_user_message(content):
    message = {"role": "user", "content": content}
    return message

def create_assistant_message(content):
    blocks = [block.model_dump(exclude_none=True) for block in content]
    message = {"role": "assistant", "content": blocks}
    return message



def run_turn(messages, system=None):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "tools":[get_current_datetime_schema,
                 get_local_timezone_schema,
                 get_system_info_schema,
                 get_disk_usage_schema],
        "thinking": {"type": "adaptive", "display": "summarized"},
    }
    
    if system:
        params["system"] = system
    
    return client.messages.create(**params)


## Tools

In [4]:
import os
import platform
import shutil

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

def get_local_timezone():
    now = datetime.now().astimezone()
    total_minutes = int(now.utcoffset().total_seconds() // 60)
    sign = "+" if total_minutes >= 0 else "-"
    hours, minutes = divmod(abs(total_minutes), 60)

    iana_name = None
    try:
        link = os.readlink("/etc/localtime")
        if "zoneinfo/" in link:
            iana_name = link.split("zoneinfo/", 1)[1]
    except OSError:
        pass

    return json.dumps({
        "iana_name": iana_name,
        "abbreviation": now.tzname(),
        "utc_offset": f"{sign}{hours:02d}:{minutes:02d}",
    })


get_local_timezone_schema = ToolParam({
    "name": "get_local_timezone",
    "description": "Returns the user's local timezone: the IANA name (e.g. 'Pacific/Honolulu'), "
                   "the abbreviation (e.g. 'HST'), and the UTC offset (e.g. '-10:00'). "
                   "Use this to interpret local times or convert between timezones.",
    "input_schema": {"type": "object", "properties": {}, "required": []},
})


def get_system_info():
    return json.dumps({
        "os": platform.system(),
        "os_version": platform.mac_ver()[0] or platform.release(),
        "machine": platform.machine(),
        "cpu_count": os.cpu_count(),
        "python_version": platform.python_version(),
        "hostname": platform.node(),
    })


get_system_info_schema = ToolParam({
    "name": "get_system_info",
    "description": "Returns basic facts about the computer this agent runs on: operating system, "
                   "OS version, CPU architecture, number of CPU cores, Python version and hostname.",
    "input_schema": {"type": "object", "properties": {}, "required": []},
})


def get_disk_usage(path="/"):
    usage = shutil.disk_usage(path)
    gb = 1024 ** 3
    return json.dumps({
        "path": path,
        "total_gb": round(usage.total / gb, 1),
        "used_gb": round(usage.used / gb, 1),
        "free_gb": round(usage.free / gb, 1),
        "percent_used": round(usage.used / usage.total * 100, 1),
    })


get_disk_usage_schema = ToolParam({
    "name": "get_disk_usage",
    "description": "Returns total, used and free disk space in GB for the disk containing the given path.",
    "input_schema": {
        "type": "object",
        "properties": {
            "path": {
                "type": "string",
                "description": "Any path on the disk to check. Defaults to '/' (the main disk).",
            }
        },
        "required": [],
    },
})


tool_functions = {
    "get_current_datetime": get_current_datetime,
    "get_local_timezone": get_local_timezone,
    "get_system_info": get_system_info,
    "get_disk_usage":get_disk_usage
    }

# Work Area

In [5]:
prompt = f""""
If my disk is more than half full, tell me the time in Tokyo, otherwise in London.
"""

system = "Use a concise narrative style when describing something.  Do not repeat text already given unless asked."
history = []
history.append(create_user_text_message(prompt))
show_history(history)

response = run_turn(history, system)


 0 user       [text] "
If my disk is more than half full, tell me the time in Tokyo, otherwise in London.



In [6]:
show_response(response)

response                                                 # Message
response.type                                            = 'message'
response.id                                              = 'msg_011CgPcZTUrat5BR8uMaCUCJ'
response.container                                       = None
response.diagnostics                                     = None
response.model                                           = 'claude-opus-5-5'
response.role                                            = 'assistant'
response.stop_details                                    = None
response.stop_reason                                     = 'tool_use'
response.stop_sequence                                   = None

response.content[0]                                      # ToolUseBlock
response.content[0].type                                 = 'tool_use'
response.content[0].id                                   = 'toolu_01KYMqTVucBNKUidyM6pZVX6'
response.content[0].input                                = {}
res

In [7]:
history.append(create_assistant_message(content=response.content))
show_history(history)

 0 user       [text] "
If my disk is more than half full, tell me the time in Tokyo, otherwise in London.

 1 assistant  [tool_use] get_disk_usage({})  id=toolu_01KYMqTVucBNKUidyM6pZVX6
              [tool_use] get_current_datetime({"date_format": "%Y-%m-%d %H:%M:%S %Z %z"})  id=toolu_01JXeGJFFz9QnZaBsH2htM9g
              [tool_use] get_local_timezone({})  id=toolu_01FGhKi9n4UHbBTm3iauM4TT


In [8]:
def run_tools (content):
    results = []
    for block in content:
        if (block.type != "tool_use"):
            continue

        print(f"calling {block.name} with {block.input} ")

        try:
            output = tool_functions[block.name](**block.input)
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(output)})
        except Exception as e:
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": f"Error: {e}", "is_error": True})

    return results



results = run_tools(response.content)
print(results)

calling get_disk_usage with {} 
calling get_current_datetime with {'date_format': '%Y-%m-%d %H:%M:%S %Z %z'} 
calling get_local_timezone with {} 
[{'type': 'tool_result', 'tool_use_id': 'toolu_01KYMqTVucBNKUidyM6pZVX6', 'content': '{"path": "/", "total_gb": 460.4, "used_gb": 152.9, "free_gb": 307.5, "percent_used": 33.2}'}, {'type': 'tool_result', 'tool_use_id': 'toolu_01JXeGJFFz9QnZaBsH2htM9g', 'content': '2026-10-25 10:49:51  '}, {'type': 'tool_result', 'tool_use_id': 'toolu_01FGhKi9n4UHbBTm3iauM4TT', 'content': '{"iana_name": "Pacific/Honolulu", "abbreviation": "HST", "utc_offset": "-10:00"}'}]


In [9]:
history.append(create_user_message(results))

In [10]:
show_history(history)

 0 user       [text] "
If my disk is more than half full, tell me the time in Tokyo, otherwise in London.

 1 assistant  [tool_use] get_disk_usage({})  id=toolu_01KYMqTVucBNKUidyM6pZVX6
              [tool_use] get_current_datetime({"date_format": "%Y-%m-%d %H:%M:%S %Z %z"})  id=toolu_01JXeGJFFz9QnZaBsH2htM9g
              [tool_use] get_local_timezone({})  id=toolu_01FGhKi9n4UHbBTm3iauM4TT
 2 user       [tool_result] for toolu_01KYMqTVucBNKUidyM6pZVX6: {"path": "/", "total_gb": 460.4, "used_gb": 152.9, "free_gb": 307.5, "percent_used": 33.2}
              [tool_result] for toolu_01JXeGJFFz9QnZaBsH2htM9g: 2026-10-25 10:49:51  
              [tool_result] for toolu_01FGhKi9n4UHbBTm3iauM4TT: {"iana_name": "Pacific/Honolulu", "abbreviation": "HST", "utc_offset": "-10:00"}


In [11]:
response = run_turn(history, system)

In [12]:
show_response(response)

response                                                 # Message
response.type                                            = 'message'
response.id                                              = 'msg_011CgPcdV6TQYKdQ78u2cjdT'
response.container                                       = None
response.diagnostics                                     = None
response.model                                           = 'claude-opus-5-5'
response.role                                            = 'assistant'
response.stop_details                                    = None
response.stop_reason                                     = 'end_turn'
response.stop_sequence                                   = None

response.content[0]                                      # ThinkingBlock
response.content[0].type                                 = 'thinking'
response.content[0].signature                            = 'CAQS/gYKEAgTGAI4AUIIdGhpbmtpbmcSDMho0wEQuPppZPQrzxoMAJQsXk0V9ACZq57jIjBse9trQ+k… (+1126 chars)
r